# 2. GitHub Actions OIDC con Vault

Este notebook configura autenticación OIDC de GitHub Actions contra Vault usando `vault` CLI para el lado de Vault y `gh` CLI para el lado de GitHub.

In [ ]:
%%bash
set -euo pipefail

# Carga variables de Vault si existen de la demo anterior.
if [ -f /tmp/vault/config.env ]; then
  set -a
  # shellcheck disable=SC1091
  source /tmp/vault/config.env
  set +a
fi

command -v gh >/dev/null
command -v vault >/dev/null
command -v jq >/dev/null

gh auth status
vault status
echo "VAULT_ADDR=${VAULT_ADDR:-not-set}"

In [ ]:
%%bash
set -euo pipefail

WORKDIR=/tmp/vault
mkdir -p ${WORKDIR}/gha

normalize_repo() {
  local raw="$1"
  raw="${raw#https://github.com/}"
  raw="${raw#git@github.com:}"
  raw="${raw%.git}"
  echo "${raw}"
}

REPO_FULL_NAME=""

# 1) Dentro de un repo git: intenta resolver con gh, y si falla usa el remote origin.
if git rev-parse --is-inside-work-tree >/dev/null 2>&1; then
  REPO_FULL_NAME="$(gh repo view --json nameWithOwner -q .nameWithOwner 2>/dev/null || true)"

  if [ -z "${REPO_FULL_NAME}" ]; then
    ORIGIN_URL="$(git remote get-url origin 2>/dev/null || true)"
    if [ -n "${ORIGIN_URL}" ]; then
      REPO_FULL_NAME="$(normalize_repo "${ORIGIN_URL}")"
    fi
  fi
fi

# 2) Fuera de git: permite usar variables de entorno estándar.
if [ -z "${REPO_FULL_NAME}" ] && [ -n "${GH_REPO:-}" ]; then
  REPO_FULL_NAME="$(normalize_repo "${GH_REPO}")"
fi
if [ -z "${REPO_FULL_NAME}" ] && [ -n "${GITHUB_REPOSITORY:-}" ]; then
  REPO_FULL_NAME="$(normalize_repo "${GITHUB_REPOSITORY}")"
fi

if [ -z "${REPO_FULL_NAME}" ] || [[ "${REPO_FULL_NAME}" != */* ]]; then
  echo "No se pudo inferir el repositorio de GitHub desde este directorio." >&2
  echo "Define GH_REPO=owner/repo y reintenta (ejemplo: export GH_REPO=hashicorp/vault-action)." >&2
  exit 1
fi

REPO_OWNER=${REPO_FULL_NAME%%/*}
REPO_NAME=${REPO_FULL_NAME##*/}
GHA_BRANCH=main
VAULT_JWT_PATH=github
VAULT_POLICY_NAME=gha-${REPO_NAME}
VAULT_ROLE_NAME=gha-${REPO_NAME}-${GHA_BRANCH}

cat > ${WORKDIR}/gha/context.env <<EOF
REPO_FULL_NAME=${REPO_FULL_NAME}
REPO_OWNER=${REPO_OWNER}
REPO_NAME=${REPO_NAME}
GHA_BRANCH=${GHA_BRANCH}
VAULT_JWT_PATH=${VAULT_JWT_PATH}
VAULT_POLICY_NAME=${VAULT_POLICY_NAME}
VAULT_ROLE_NAME=${VAULT_ROLE_NAME}
EOF

cat ${WORKDIR}/gha/context.env

## Policy de Vault para GitHub Actions

La policy de ejemplo permite leer secretos en `secret/data/gha/*` (KV v2). Ajusta paths/capabilities según tu caso.

In [ ]:
%%bash
set -euo pipefail
source /tmp/vault/gha/context.env

cat > /tmp/vault/gha/${VAULT_POLICY_NAME}.hcl <<EOF
path "secret/data/gha/*" {
  capabilities = ["read"]
}

path "secret/metadata/gha/*" {
  capabilities = ["read", "list"]
}
EOF

vault policy write ${VAULT_POLICY_NAME} /tmp/vault/gha/${VAULT_POLICY_NAME}.hcl
vault policy read ${VAULT_POLICY_NAME}

In [ ]:
%%bash
set -euo pipefail
source /tmp/vault/gha/context.env

vault auth enable -path=${VAULT_JWT_PATH} jwt || true

vault write auth/${VAULT_JWT_PATH}/config \
  oidc_discovery_url="https://token.actions.githubusercontent.com" \
  bound_issuer="https://token.actions.githubusercontent.com"

vault read auth/${VAULT_JWT_PATH}/config

In [ ]:
%%bash
set -euo pipefail
source /tmp/vault/gha/context.env

cat > /tmp/vault/gha/role-${VAULT_ROLE_NAME}.json <<EOF
{
  "role_type": "jwt",
  "user_claim": "actor",
  "bound_audiences": "https://github.com/${REPO_OWNER}",
  "bound_claims_type": "glob",
  "bound_claims": {
    "repository": "${REPO_FULL_NAME}",
    "ref": "refs/heads/${GHA_BRANCH}"
  },
  "token_policies": "${VAULT_POLICY_NAME}",
  "token_ttl": "1h"
}
EOF

vault write auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME} @/tmp/vault/gha/role-${VAULT_ROLE_NAME}.json
vault read auth/${VAULT_JWT_PATH}/role/${VAULT_ROLE_NAME}

In [ ]:
%%bash
set -euo pipefail

# Secret de ejemplo para validar lectura desde GHA.
vault kv put secret/gha/demo api_key="$(openssl rand -hex 16)"
vault kv get secret/gha/demo

## Workflow de ejemplo en GitHub Actions

Este job solicita un token OIDC (`id-token: write`), autentica contra Vault y lee un secreto.

Requisitos en el repositorio de GitHub:
- Variable `VAULT_ADDR` (URL de Vault, por ejemplo `https://...`)
- Variable `VAULT_AUTH_PATH` (en este ejemplo `github`)
- Variable `VAULT_AUTH_ROLE` (rol creado en este notebook)

In [ ]:
%%bash
set -euo pipefail
source /tmp/vault/gha/context.env

WORKFLOW_FILE=.github/workflows/vault-oidc.yml
mkdir -p .github/workflows
cat > ${WORKFLOW_FILE} <<'EOF'
name: vault-oidc
on:
  workflow_dispatch:
jobs:
  read-secret:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      id-token: write
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Import Vault Secrets
        id: vault
        uses: hashicorp/vault-action@v3
        with:
          url: ${{ vars.VAULT_ADDR }}
          method: jwt
          path: ${{ vars.VAULT_AUTH_PATH }}
          role: ${{ vars.VAULT_AUTH_ROLE }}
          secrets: |
            secret/data/gha/demo api_key | API_KEY

      - name: Use Secret
        run: |
          test -n "${API_KEY}"
          echo "Secret loaded successfully"
EOF

gh variable set VAULT_AUTH_PATH --repo "${REPO_FULL_NAME}" --body "${VAULT_JWT_PATH}"
gh variable set VAULT_AUTH_ROLE --repo "${REPO_FULL_NAME}" --body "${VAULT_ROLE_NAME}"

# Si VAULT_ADDR está disponible localmente, publícalo también como variable del repo.
if [ -n "${VAULT_ADDR:-}" ]; then
  gh variable set VAULT_ADDR --repo "${REPO_FULL_NAME}" --body "${VAULT_ADDR}"
fi

echo "Workflow generado localmente en ${WORKFLOW_FILE}"
echo "Sube el workflow al repo ${REPO_FULL_NAME} con git push o gh api repos/.../contents si prefieres API."
echo "Variables GHA configuradas: VAULT_AUTH_PATH, VAULT_AUTH_ROLE${VAULT_ADDR:+, VAULT_ADDR}"